# A2.8 · Just-in-time authority

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

Builds on **[A2.7 · Systems that don't understand agents](https://spbreed.github.io/cyber-commons/lessons/A2.7.html)**.

| | |
|---|---|
| Open-source tooling | Keycloak, Vault (OSS) |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Every control so far has narrowed *what* an identity may hold. Just-in-time
authority narrows *when*.

The default is a **standing grant**: `deploy-agent` holds `deploy:prod`
permanently, because it needs it sometimes. The audit question that produces is
"who has deploy:prod?" — answered by the same dull list every quarter, which
tells a reviewer nothing.

JIT replaces it with authority that exists only for the duration of one
justified task. The audit question becomes "who held it, for what, and for how
long?" — which is answerable, sampleable, and genuinely interesting, because
each grant carries a reason a human can dispute.

The two design decisions that matter:

- **TTL.** Derived from the measured duration of the real task, not a round
  number. Too short and the agent fails mid-task; too long and you have
  reinvented the standing grant with extra steps.
- **What happens at expiry.** The agent must handle losing authority gracefully.
  An agent that crashes when its grant expires will be given a longer TTL, and
  then a permanent one.

## 2 · Demo — standing grants, and what an auditor sees

The starting position, and the reason it is unsatisfying.

In [ ]:
import time
from dataclasses import dataclass, field

STANDING = {
    "deploy-agent":  {"deploy:prod", "artifact:read"},
    "patch-agent":   {"repo:write", "repo:read"},
    "backup-runner": {"db:read", "s3:write"},
}
print("standing grants — the quarterly access review:")
for actor, scopes in STANDING.items():
    print(f"   {actor:16s} {sorted(scopes)}")
print("\nThe reviewer's only possible question: 'should this still exist?'")
print("With no usage data attached, the honest answer is always 'probably'.")

# how often is that authority actually exercised?
USAGE = {"deploy-agent": 6, "patch-agent": 210, "backup-runner": 30}   # per 90 days
print("\nactual use in the last 90 days:")
for actor, n in USAGE.items():
    held_seconds = 90 * 86400
    used_seconds = n * 180          # ~3 minutes of real work per use
    print(f"   {actor:16s} used {n:>3}×  → authority idle "
          f"{100 * (1 - used_seconds/held_seconds):.2f}% of the time")

## 3 · Where it breaks

`deploy-agent` holds production deploy rights continuously and uses them six times a quarter. For 99.99% of its life, the credential is a liability with no corresponding benefit — and that idle window is exactly when a compromise would go unnoticed, because nobody is watching a capability that is not being used.

## 4 · The control — grants with a reason and an expiry

In [ ]:
class GrantExpired(Exception): pass

@dataclass
class JITGrant:
    actor: str
    scope: str
    reason: str                 # free text, but MANDATORY — this is the audit value
    ttl: float
    granted: float = field(default_factory=time.time)
    used: int = 0

    @property
    def active(self): return time.time() - self.granted < self.ttl
    @property
    def age(self): return time.time() - self.granted

    def use(self):
        if not self.active:
            raise GrantExpired(f"{self.actor}'s {self.scope} grant expired "
                               f"after {self.ttl}s ({self.reason!r})")
        self.used += 1
        return True

    def audit_line(self):
        return (f"{self.actor:14s} {self.scope:14s} "
                f"{'ACTIVE' if self.active else 'expired':8s} "
                f"ttl={self.ttl:>5.1f}s used={self.used}  reason={self.reason!r}")

grants = [
    JITGrant("deploy-agent", "deploy:prod", "roll out fix for CVE-2026-1188", ttl=0.6),
    JITGrant("patch-agent",  "repo:write",  "patch finding-4471",             ttl=60),
]
print("at issue:")
for g in grants: print("   " + g.audit_line())

grants[0].use(); grants[1].use()
time.sleep(0.7)

print("\nafter the deploy window closes:")
for g in grants: print("   " + g.audit_line())

## 5 · Verify — the agent must survive expiry

This is the design decision that decides whether JIT survives contact with an on-call engineer. An agent that crashes on expiry gets a longer TTL; an agent that re-requests with a reason keeps the control alive.

In [ ]:
def naive_agent(grant, steps):
    """Crashes when authority disappears mid-task."""
    for i in range(steps):
        grant.use()                      # raises when expired
        time.sleep(0.25)
    return "completed"

def resilient_agent(grant_factory, grant, steps, reason):
    """Re-requests, with a reason, and records why. Survives expiry."""
    events = []
    for i in range(steps):
        try:
            grant.use()
        except GrantExpired:
            events.append(f"step {i}: grant expired → re-requesting")
            grant = grant_factory(reason=f"{reason} (continuation, step {i})")
            grant.use()
        events.append(f"step {i}: ok")
        time.sleep(0.25)
    return events, grant

g = JITGrant("deploy-agent", "deploy:prod", "rollout", ttl=0.3)
try:
    naive_agent(g, 4)
except GrantExpired as e:
    print("naive agent    :", e)
    print("                 → on-call asks for a 24h TTL, and JIT is over.")

factory = lambda reason: JITGrant("deploy-agent", "deploy:prod", reason, ttl=0.3)
events, final = resilient_agent(factory, factory("rollout"), 4, "rollout")
print("\nresilient agent:")
for e in events: print("   ", e)
print("   final grant:", final.audit_line())
print("\nThe audit trail now contains every continuation and its reason —")
print("which is strictly more information than a standing grant ever produced.")

## What you just proved

The standing-grant review shows `deploy-agent`'s authority idle 99.99% of the time. The JIT grants print ACTIVE at issue and the 0.6-second one shows expired afterwards, each carrying its reason. The naive agent raises `GrantExpired` mid-task; the resilient agent detects expiry, re-requests with a continuation reason, and completes all four steps.

## Your turn

Pick the standing grant with the worst ratio of held-time to used-time in your estate. Measure how long the real task takes, set the TTL at the 95th percentile of that, and make the agent re-request. The reason field is where the audit value lives — insist it be specific.

---

**Next → [A2.9 · The classic failures](https://spbreed.github.io/cyber-commons/lessons/A2.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*